# **About Dataset**

**Context**

Predict next-day rain by training classification models on the target variable RainTomorrow.

**Content**

This dataset contains about 10 years of daily weather observations from many locations across Australia.

RainTomorrow is the target variable to predict. It means -- did it rain the next day, Yes or No? This column is Yes if the rain for that day was 1mm or more.

# **Imports**

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split , cross_val_score , RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from category_encoders import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score , classification_report , accuracy_score , confusion_matrix 

In [ ]:
df = pd.read_csv('/kaggle/input/weather-dataset-rattle-package/weatherAUS.csv')
df.head()

# **Explore the Dataset**

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# describtion for numerical columns
df.describe()

# **Deal with missing values**

In [ ]:
# calculating missing values
(df.isna().sum().sort_values(ascending = False)) / len(df)

In [ ]:
# Categorical columns that have missing values
df_cat = df.select_dtypes(include = ['object'])

(df_cat.isna().sum().sort_values(ascending = False)) / len(df)

> we will drop the rows that have a missing values because the number of rows with missing values is relatively small

In [ ]:
# Drop the rows with the missing values
for column_name in df_cat.columns:
    df.dropna(subset=[column_name], inplace=True)
    df_cat.dropna(subset=[column_name], inplace=True)

# Check the shape after dropping rows with missing values
df.shape

In [ ]:
# numerical columns that have missing values
df_num = df.select_dtypes(include = ['float64', 'int64'])

(df_num.isna().sum().sort_values(ascending = False)) / len(df)

> these columns we will filling it by SimpleImputer method in the pipeLine

# # **Data distribution**

# **Numerical columns**

In [ ]:
df_num.hist(figsize=(16, 20), bins=40, xlabelsize=6, ylabelsize=6);

In [ ]:
# Encoding our target column
replacement_dict = {'Yes': 1, 'No': 0}
df['RainTomorrow'] = df['RainTomorrow'].replace(replacement_dict)

df_num = pd.concat([df_num, df['RainTomorrow']], axis=1)

In [ ]:
fig, axes = plt.subplots(nrows=len(df_num.columns) // 2, ncols=2, figsize=(13, 20))

for idx, column in enumerate(df_num.drop(columns = 'RainTomorrow')):
    row_idx = idx // 2
    col_idx = idx % 2
    
    sns.kdeplot(df[df["RainTomorrow"] == 1][column], alpha=0.5, fill=True, color="#000CEB", label="RainTomorrow", ax=axes[row_idx, col_idx])
    sns.kdeplot(df[df["RainTomorrow"] == 0][column], alpha=0.5, fill=True, color="#97B9F4", label="RainTomorrow", ax=axes[row_idx, col_idx])
    
    axes[row_idx, col_idx].set_xlabel(column)
    axes[row_idx, col_idx].set_ylabel("Frequency")
    axes[row_idx, col_idx].set_title(f"{column} Distribution over Rain Tomorrow column")
    axes[row_idx, col_idx].legend()

plt.tight_layout()
plt.show()

# **Categorical columns**

**Check for high and low cardinality**

In [ ]:
df_cat.nunique().sort_values()

> We will drop Date column cecause it is a high cardinality

In [ ]:
df_cat.drop(columns = 'Date', inplace = True)

# **Outliers**

In [ ]:
# Outlier Equation
def outlier_thresholds (dataframe, column, q1=0.25, q3=0.75) :
    quartile1 = dataframe[column].quantile(q1)
    quartile3 = dataframe[column].quantile(q3)
    interquartile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquartile_range
    low_limit = quartile1 - 1.5 * interquartile_range
    return low_limit, up_limit

# Checking Outliers
def check_outlier(dataframe, column):
    low_limit, up_limit = outlier_thresholds(dataframe, column)
    outliers = (dataframe[column] > up_limit) | (dataframe[column] < low_limit)
    if outliers.any():
        return True
    else:
        return False

# Removing the outliers
def replace_with_thresholds (dataframe, column) :
    low_limit , up_limit = outlier_thresholds(dataframe, column)
    dataframe.loc[(dataframe[column] < low_limit), column] = low_limit
    dataframe.loc[(dataframe[column] > up_limit), column] = up_limit

**Plotting Outliers**

In [ ]:
fig, axes = plt.subplots(nrows=len(df_num.columns) // 2, ncols=2, figsize=(16, 30))

for idx, column in enumerate(df_num.drop(columns = 'RainTomorrow')):
    row_idx = idx // 2
    col_idx = idx % 2
    
    sns.boxenplot( x='RainTomorrow' , y= column , data=df, ax=axes[row_idx, col_idx])
    
    axes[row_idx, col_idx].set_xlabel("RainTomorrow")
    axes[row_idx, col_idx].set_ylabel(column)
    axes[row_idx, col_idx].set_title(f"{column} Distribution")

plt.tight_layout()
plt.show()

# **Removing outliers**

In [ ]:
print('After removing the outliers :')
for column in (df_num.drop(columns = 'RainTomorrow').columns):
    replace_with_thresholds(df_num, column)
    print(column, check_outlier(df_num, column))

# **Multicollinearity**

In [ ]:
corr = df_num.drop(columns= 'RainTomorrow').corr()
fig , ax = plt.subplots(figsize=(15 , 10))
sns.heatmap(corr ,annot= True , ax=ax , cmap= 'Greens');

**Now we will search for columns that has a correlation more than 70% and drop one of them with the condition that the correlation with the target column (HeartDisease) is smaller than another column**

In [ ]:
# check the correlation for columns => MinTemp & MaxTemp with the target
print(f"Correlation between MinTemp and MaxTemp :{df_num['MinTemp'].corr(df_num['MaxTemp'])}")

print(f"Correlation between MinTemp and the target :{df_num['MinTemp'].corr(df_num['RainTomorrow'])}")

print(f"Correlation between MaxTemp and the target :{df_num['MaxTemp'].corr(df_num['RainTomorrow'])}")

In [ ]:
# check the correlation for columns => Temp3pm & Temp9am with the target
print(f"Correlation between Temp3pm and Temp9am :{df_num['Temp3pm'].corr(df_num['Temp9am'])}")

print(f"Correlation between Temp3pm and the target :{df_num['Temp3pm'].corr(df_num['RainTomorrow'])}")

print(f"Correlation between Temp9am and the target :{df_num['Temp9am'].corr(df_num['RainTomorrow'])}")

In [ ]:
# drop 'MinTemp' and 'Temp9am' column
df_num.drop(columns = ['MinTemp','Temp9am'], inplace = True)

# **Target balance**

In [ ]:
# Recreate our dataframe
df_cat.drop(columns = 'RainTomorrow', inplace = True)
df = pd.concat([df_num, df_cat], axis=1)

In [ ]:
target_column = df.RainTomorrow.value_counts()

# pie chart for target column
plt.pie(target_column, labels = target_column.index, autopct="%1.1f%%", explode = [0,0.1], colors = ["#0142F4","#00EBB5"])
plt.title("balance of Target column")
plt.axis("equal")
plt.show()

> Our data is balanced

# **Splitting data for train and test**

In [ ]:
X = df.drop(columns = 'RainTomorrow')
target = df['RainTomorrow']

X_train , X_test , y_train , y_test = train_test_split(X ,target ,test_size=0.2 , random_state=42 )
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# **Baseline**

In [ ]:
dummy_classifier = DummyClassifier(strategy = 'stratified', random_state = 42) 
dummy_classifier.fit(X_train, y_train) 
y_pred = dummy_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline Model Accuracy: {accuracy:.4f}")

# **Modeling**

In [ ]:
def train(classifier,x_train,y_train,x_test,y_test):
    
    classifier.fit(x_train,y_train)
    prediction = classifier.predict(x_test)
    cv = RepeatedStratifiedKFold(n_splits = 10,n_repeats = 3,random_state = 42)
    print("Cross Validation Score : ",'{0:.2%}'.format(cross_val_score(classifier,x_train,y_train,cv = cv,scoring = 'roc_auc').mean()))
    

def model_evaluation(classifier,x_test,y_test):
    
    # Confusion Matrix
    cm = confusion_matrix(y_test,classifier.predict(x_test))
    names = ['True Neg','False Pos','False Neg','True Pos']
    counts = [value for value in cm.flatten()]
    percentages = ['{0:.2%}'.format(value) for value in cm.flatten()/np.sum(cm)]
    labels = [f'{v1}\n{v2}\n{v3}' for v1, v2, v3 in zip(names,counts,percentages)]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cm,annot = labels,cmap = 'Greens',fmt ='')
    
    # Classification Report
    print(classification_report(y_test,classifier.predict(x_test)))

# Random forest

In [ ]:
rf_classifier = make_pipeline(
    OneHotEncoder(),
    SimpleImputer(strategy='mean'),
    MinMaxScaler(),
    RandomForestClassifier(n_estimators=25, random_state=42)
)

In [ ]:
# Training the model
train(rf_classifier, X_train, y_train, X_test, y_test)

# Evaluate the model
model_evaluation(rf_classifier, X_test, y_test)

# XGBoost

In [ ]:
xgb_classifier = make_pipeline(
    OneHotEncoder(),
    SimpleImputer(strategy='mean'),
    MinMaxScaler(),
    xgb.XGBClassifier(objective = 'binary:logistic', random_state = 42)
)

In [ ]:
# Training the model
train(xgb_classifier, X_train, y_train, X_test, y_test)

# Evaluate the model
model_evaluation(xgb_classifier, X_test, y_test)